# Chapitre 8 - Cours 3 : horloges non fiables

Les applications posent sans cesse des questions de temps : cette requête a-t-elle
expiré ? quand ce cache périme-t-il ? qui a écrit en dernier ? Or chaque machine a sa
propre horloge, un quartz imparfait, synchronisé tant bien que mal par le réseau. Ce
notebook décortique les deux sortes d'horloges, ce que vaut leur synchronisation, et les
dangers de s'y fier pour **ordonner** des événements. Il couvre la section *Unreliable
Clocks* : *Monotonic Versus Time-of-Day Clocks*, *Clock Synchronization and Accuracy* et
*Relying on Synchronized Clocks* (*Timestamps for ordering events*, *Clock readings have a confidence interval*, *Synchronized clocks for global snapshots*).

**Plan.**

1. Mesurer une durée, dater un instant.
2. Horloge calendaire et horloge monotone.
3. Synchronisation et précision : dérive, NTP, secondes intercalaires.
4. Horodater pour ordonner : le piège de « la dernière écriture gagne ».
5. Lectures d'horloge avec intervalle de confiance, et instantanés globaux.

In [1]:
import math
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import FancyBboxPatch

from ddia import distsys as ds
from ddia.plotting import (AMBER, GRAY, GREEN, INDIGO, LIGHT, ROSE, SKY, SLATE, TEAL, VIOLET,
                           annotate_bars, set_style, subtitle)
from ddia.requirements import plain_log_axis
from ddia.sim import Clock, Network, Node
from ddia.timelines import SpaceTime

set_style()
plt.rcParams["axes.titlepad"] = 16
pd.set_option("display.precision", 4)
rng = np.random.default_rng(0)

## 1. Mesurer une durée, dater un instant (section *Unreliable Clocks*)

Le livre liste huit questions typiques ; elles se répartissent en deux familles.

| question | nature |
|---|---|
| cette requête a-t-elle expiré ? | **durée** |
| quel est le p99 du temps de réponse ? | **durée** |
| combien de requêtes par seconde ces cinq dernières minutes ? | **durée** |
| combien de temps l'utilisateur est-il resté sur le site ? | **durée** |
| quand cet article a-t-il été publié ? | **instant** |
| à quelle date et quelle heure envoyer le courriel de rappel ? | **instant** |
| quand cette entrée du cache expire-t-elle ? | **instant** |
| quel est l'horodatage de ce message d'erreur dans le journal ? | **instant** |

Dans un système distribué, le temps est délicat pour deux raisons : la communication
n'est pas instantanée (un message est reçu **après** son envoi, mais on ne sait pas
**combien** après, cours 2), et chaque machine a sa propre horloge matérielle,
généralement un **oscillateur à quartz**, légèrement trop rapide ou trop lent. Le
protocole **NTP** (Network Time Protocol) permet de recaler les horloges sur des
serveurs, eux-mêmes calés sur une source plus précise (récepteur GPS).

Python expose les deux sortes d'horloges des systèmes d'exploitation. Interrogeons-les
sur cette machine :

In [2]:
infos = []
for nom, role in [("time", "calendaire (time.time)"), ("monotonic", "monotone (time.monotonic)"),
                  ("perf_counter", "monotone haute résolution (time.perf_counter)")]:
    i = time.get_clock_info(nom)
    infos.append(dict(horloge=role, implémentation=i.implementation, résolution_s=i.resolution,
                      ajustable=i.adjustable, monotone=i.monotonic))
pd.DataFrame(infos).set_index("horloge")

,implémentation,résolution_s,ajustable,monotone
horloge,,,,
calendaire (time.time),clock_gettime(CLOCK_REALTIME),1.0000e-09,True,False
monotone (time.monotonic),clock_gettime(CLOCK_MONOTONIC),1.0000e-09,False,True
monotone haute résolution (time.perf_counter),clock_gettime(CLOCK_MONOTONIC),1.0000e-09,False,True


Sous Linux, `time.time()` lit `CLOCK_REALTIME` : elle est **ajustable** (NTP ou un
administrateur peuvent la modifier) et **non monotone**. `time.monotonic()` lit
`CLOCK_MONOTONIC` : jamais ajustée par sauts, toujours croissante. La résolution affichée
(la nanoseconde) ne dit rien de l'**exactitude**, comme on va le voir.

> **À retenir.** Une durée et un instant sont deux questions différentes, qui
> appellent deux horloges différentes. Aucune des deux n'est partagée entre machines.